In [1]:
# --- Reward component budget pass for Connect4Env ---
# Samples random midgame states, evaluates all legal moves, and reports component ranges.
# Goal: confirm no shaping term dominates purely by scale (unless I *want* it to).

import numpy as np
import pandas as pd

from C4.connect4_env import (
    Connect4Env,
    STRIDE, FULL_MASK, CENTER_COL,
    _bb_has_won, _sum_pure_weighted,
    WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
)

from tqdm.auto import tqdm

def _eval_action_components(env: Connect4Env, action: int):
    """Compute reward components for a candidate action WITHOUT mutating env."""
    c = int(action)
    if c < 0 or c >= env.COLS or env._heights[c] >= env.ROWS:
        return None

    mover = int(env.current_player)

    # snapshot before move
    mask_before = np.uint64(env._mask)
    pos1_before  = np.uint64(env._pos1)
    pos2_before  = np.uint64(env._pos2)
    heights_before = env._heights.copy()

    mover_bb_before = pos1_before if mover == 1 else pos2_before
    opp_bb_before   = pos2_before if mover == 1 else pos1_before

    opp_immediate_before = env._count_immediate_wins_bits(int(opp_bb_before), int(mask_before))

    # apply move (locally)
    r_bot = int(heights_before[c])
    bit = np.uint64(env._bit_at(c, r_bot))
    mask_after = mask_before | bit

    if mover == 1:
        pos1_after = pos1_before | bit
        pos2_after = pos2_before
    else:
        pos2_after = pos2_before | bit
        pos1_after = pos1_before

    heights_after = heights_before.copy()
    heights_after[c] = r_bot + 1

    mover_bb_after = pos1_after if mover == 1 else pos2_after
    opp_bb_after   = pos2_after if mover == 1 else pos1_after

    # terminal?
    terminal = False
    winner = None
    if _bb_has_won(np.uint64(mover_bb_after), np.int32(STRIDE)):
        terminal = True
        winner = mover
    elif mask_after == np.uint64(FULL_MASK):
        terminal = True
        winner = 0

    # Base record (include state context for debugging)
    rec = {
        "ply": int(env.ply),
        "mover": mover,
        "action": c,
        "r_bot": r_bot,
        "terminal": terminal,
        "winner": winner if terminal else None,
    }

    if terminal:
        # Terminal rewards in your env bypass shaping (as in step()).
        if winner == mover:
            total_raw = float(env.WIN_REWARD)
        elif winner == 0:
            total_raw = float(env.DRAW_REWARD)
        else:
            total_raw = float(env.LOSS_PENALTY)

        rec.update({
            "threat_reward": 0.0,
            "block_reward": 0.0,
            "fork_bonus": 0.0,
            "block_fork_bonus": 0.0,
            "center_reward": 0.0,
            "parity_reward": 0.0,
            "tempo_reward": 0.0,
            "threatspace_reward": 0.0,
            "immediate_loss_penalty": 0.0,
            "step_penalty": 0.0,
            "total_raw": total_raw,
            "total_clipped": float(np.clip(total_raw, -env.MAX_REWARD, env.MAX_REWARD)),
            "total_final": total_raw,  # terminal path does not subtract STEP_PENALTY in your code
            "clipped": abs(total_raw) > float(env.MAX_REWARD),
        })
        return rec

    # ---------- Floating-aware threat shaping ----------
    threat2 = _sum_pure_weighted(
        np.uint64(mover_bb_after),
        np.uint64(opp_bb_after),
        np.uint64(mask_after),
        heights_after,
        np.int32(2),
        WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
        float(env.FLOATING_NEAR),
        float(env.FLOATING_FAR),
        float(env.VERT_MUL),
    )
    threat3 = _sum_pure_weighted(
        np.uint64(mover_bb_after),
        np.uint64(opp_bb_after),
        np.uint64(mask_after),
        heights_after,
        np.int32(3),
        WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
        float(env.FLOATING_NEAR),
        float(env.FLOATING_FAR),
        float(env.VERT_MUL),
    )

    # Opponent threats (before/after) -> "block" deltas (floating-aware)
    opp2_before = _sum_pure_weighted(
        np.uint64(opp_bb_before),
        np.uint64(mover_bb_before),
        np.uint64(mask_before),
        heights_before,
        np.int32(2),
        WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
        float(env.FLOATING_NEAR),
        float(env.FLOATING_FAR),
        float(env.VERT_MUL),
    )
    opp3_before = _sum_pure_weighted(
        np.uint64(opp_bb_before),
        np.uint64(mover_bb_before),
        np.uint64(mask_before),
        heights_before,
        np.int32(3),
        WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
        float(env.FLOATING_NEAR),
        float(env.FLOATING_FAR),
        float(env.VERT_MUL),
    )
    opp2_after = _sum_pure_weighted(
        np.uint64(opp_bb_after),
        np.uint64(mover_bb_after),
        np.uint64(mask_after),
        heights_after,
        np.int32(2),
        WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
        float(env.FLOATING_NEAR),
        float(env.FLOATING_FAR),
        float(env.VERT_MUL),
    )
    opp3_after = _sum_pure_weighted(
        np.uint64(opp_bb_after),
        np.uint64(mover_bb_after),
        np.uint64(mask_after),
        heights_after,
        np.int32(3),
        WIN_MASKS, WIN_B, WIN_C, WIN_R, WIN_KIND,
        float(env.FLOATING_NEAR),
        float(env.FLOATING_FAR),
        float(env.VERT_MUL),
    )

    block2 = float(max(0.0, float(opp2_before - opp2_after)))
    block3 = float(max(0.0, float(opp3_before - opp3_after)))

    threat_reward = (float(env.THREAT2_VALUE) * float(threat2)) + (float(env.THREAT3_VALUE) * float(threat3))
    block_reward  = (float(env.BLOCK2_VALUE)  * float(block2))  + (float(env.BLOCK3_VALUE)  * float(block3))

    # ---------- Center shaping ----------
    center_reward = float(env.CENTER_REWARD) * float(env._CENTER_WEIGHTS_ARR[c])
    if c == CENTER_COL and r_bot == 0:
        opening_decay = float(np.exp(-env.ply / env.OPENING_DECAY_STEPS))
        bonus = float(env.CENTER_REWARD_BOTTOM) * (2.0 if env.ply == 0 else opening_decay)
        center_reward += bonus

    # ---------- Immediate wins / fork-ish ----------
    my_immediate_after  = env._count_immediate_wins_bits(int(mover_bb_after), int(mask_after))
    opp_immediate_after = env._count_immediate_wins_bits(int(opp_bb_after),   int(mask_after))

    fork_bonus = float(env.FORK_BONUS) if my_immediate_after >= 2 else 0.0
    blocked_fork = (opp_immediate_before >= 2) and (opp_immediate_after < opp_immediate_before)
    block_fork_bonus = float(env.BLOCK_FORK_BONUS) if blocked_fork else 0.0

    immediate_loss_penalty = float(env.OPP_IMMEDIATE_PENALTY) * float(opp_immediate_after)

    # ---------- Parity shaping (enabled only if D1 occupied) ----------
    parity_reward = 0.0
    role_first_mark, parity_enabled = env._role_first_from_center(np.uint64(mask_after), np.uint64(pos1_after), np.uint64(pos2_after))
    if parity_enabled:
        mover_is_first_role = (mover == int(role_first_mark))
        prefer_par = 0 if mover_is_first_role else 1
        opp_prefer_par = 1 if mover_is_first_role else 0

        if (r_bot & 1) == prefer_par:
            parity_reward += float(env.PARITY_MOVE_BONUS)
        else:
            parity_reward -= float(env.PARITY_MOVE_BONUS)

        if r_bot + 1 < env.ROWS:
            if ((r_bot + 1) & 1) == opp_prefer_par:
                parity_reward -= float(env.PARITY_UNLOCK_PENALTY)

    # ---------- Tempo squeeze (proxy: reduce opponent safe moves) ----------
    tempo_reward = 0.0
    if float(env.TEMPO_SQUEEZE_W) != 0.0:
        opp_safe = int(
            env._N["count_safe_moves"](
                np.uint64(opp_bb_after),
                np.uint64(mask_after),
                env._N["CENTER_ORDER"],
                env._N["TOP_MASK"],
                env._N["BOTTOM_MASK"],
                env._N["COL_MASK"],
                np.int32(STRIDE),
            )
        )
        tempo_reward = float(env.TEMPO_SQUEEZE_W) * float(max(0, 7 - opp_safe))

    # ---------- Threat-space ----------
    threatspace_reward = float(env.THREATSPACE_W) * float(my_immediate_after) if float(env.THREATSPACE_W) != 0.0 else 0.0

    # Total
    total_raw = (
        float(threat_reward)
        + float(block_reward)
        + float(fork_bonus)
        + float(block_fork_bonus)
        + float(center_reward)
        + float(parity_reward)
        + float(tempo_reward)
        + float(threatspace_reward)
        - float(immediate_loss_penalty)
    )

    total_clipped = float(np.clip(total_raw, -env.MAX_REWARD, env.MAX_REWARD))
    total_final = float(total_clipped - float(env.STEP_PENALTY))

    rec.update({
        "threat_reward": float(threat_reward),
        "block_reward": float(block_reward),
        "fork_bonus": float(fork_bonus),
        "block_fork_bonus": float(block_fork_bonus),
        "center_reward": float(center_reward),
        "parity_reward": float(parity_reward),
        "tempo_reward": float(tempo_reward),
        "threatspace_reward": float(threatspace_reward),
        "immediate_loss_penalty": float(immediate_loss_penalty),
        "step_penalty": float(env.STEP_PENALTY),
        "total_raw": float(total_raw),
        "total_clipped": float(total_clipped),
        "total_final": float(total_final),
        "clipped": abs(float(total_raw)) > float(env.MAX_REWARD),
        "parity_enabled": bool(parity_enabled),
        "my_immediate_after": int(my_immediate_after),
        "opp_immediate_after": int(opp_immediate_after),
    })
    return rec


def _sample_midgame_env(rng: np.random.Generator, min_ply=8, max_ply=26, max_tries=200):
    """Create a random midgame nonterminal env by playing random legal moves."""
    for _ in range(max_tries):
        env = Connect4Env()
        env.reset()
        target = int(rng.integers(min_ply, max_ply + 1))

        while (env.ply < target) and (not env.done):
            legal = env.available_actions()
            env.step(int(rng.choice(legal)))

        if (not env.done) and (env.ply >= min_ply):
            return env
    return None


def _summarize_ranges(df: pd.DataFrame, cols):
    rows = []
    for col in cols:
        s = df[col].astype(float)
        rows.append({
            "term": col,
            "mean": s.mean(),
            "std": s.std(ddof=1),
            "min": s.min(),
            "p05": s.quantile(0.05),
            "p25": s.quantile(0.25),
            "median": s.quantile(0.50),
            "p75": s.quantile(0.75),
            "p95": s.quantile(0.95),
            "max": s.max(),
            "mean_abs": s.abs().mean(),
        })
    out = pd.DataFrame(rows).set_index("term").sort_values("mean_abs", ascending=False)
    return out


# -------------------- RUN THE PASS --------------------
SEED = 123
N_STATES = 1000            # increase for tighter estimates
MIN_PLY, MAX_PLY = 10, 26  # "midgame-ish"
EVAL_ALL_LEGAL = True      # else samples 1 random legal move per state

rng = np.random.default_rng(SEED)

records = []
envs = []
for _ in tqdm(range(N_STATES), desc="Sampling states"):
    e = _sample_midgame_env(rng, min_ply=MIN_PLY, max_ply=MAX_PLY)
    if e is not None:
        envs.append(e)

# Warm up numba once (avoids first-call skew on timings)
if envs:
    e0 = envs[0]
    _ = _eval_action_components(e0, e0.available_actions()[0])

for e in tqdm(envs, desc="Evaluating actions"):
    legal = e.available_actions()
    if not legal:
        continue
    acts = legal if EVAL_ALL_LEGAL else [int(rng.choice(legal))]
    for a in acts:
        rec = _eval_action_components(e, a)
        if rec is not None:
            records.append(rec)

df = pd.DataFrame(records)
print("Samples:", len(df), "| States:", len(envs))
if "parity_enabled" in df.columns:
    print("Parity enabled rate:", float(df["parity_enabled"].mean()))
print("Clip rate (nonterminal):", float(df.loc[~df["terminal"], "clipped"].mean()))
print("Terminal rate:", float(df["terminal"].mean()))

# Focus on NONTERMINAL shaping signals (terminal rewards are their own planet)
df_nt = df.loc[~df["terminal"]].copy()

terms = [
    "threat_reward",
    "block_reward",
    "fork_bonus",
    "block_fork_bonus",
    "center_reward",
    "parity_reward",
    "tempo_reward",
    "threatspace_reward",
    "immediate_loss_penalty",
    "step_penalty",
    "total_raw",
    "total_clipped",
    "total_final",
]

# Range table
budget = _summarize_ranges(df_nt, terms)
display(budget)

# "Dominance" check: mean(|term|) relative to mean(|total_raw|)
den = float(df_nt["total_raw"].abs().mean()) + 1e-9
dominance = (budget["mean_abs"] / den).rename("mean_abs / mean_abs(total_raw)")
display(dominance.to_frame().sort_values("mean_abs / mean_abs(total_raw)", ascending=False))

# Optional: show how often each term is basically zero (sparsity)
sparsity = ((df_nt[terms].abs() < 1e-9).mean()).rename("fraction_zero")
display(sparsity.to_frame().sort_values("fraction_zero", ascending=False))


Sampling states:   0%|          | 0/1000 [00:00<?, ?it/s]

Evaluating actions:   0%|          | 0/1000 [00:00<?, ?it/s]

Samples: 6735 | States: 1000
Parity enabled rate: 0.914585652934627
Clip rate (nonterminal): 0.0003181167488468268
Terminal rate: 0.06651818856718635


,mean,std,min,p05,p25,median,p75,p95,max,mean_abs
term,,,,,,,,,,
immediate_loss_penalty,517.735009,690.116184,0.0000,0.00000,0.000,0.0000,1000.00000,2000.00000,4000.000000,517.735009
total_raw,-422.245891,679.892171,-3978.5625,-1911.03125,-916.365,7.4475,82.07250,291.78750,603.875000,514.724963
total_clipped,-422.103186,679.198402,-3500.0000,-1911.03125,-916.365,7.4475,82.07250,291.78750,603.875000,514.582258
total_final,-422.203186,679.198402,-3500.1000,-1911.13125,-916.465,7.3475,81.97250,291.68750,603.775000,514.566018
threat_reward,44.445980,42.086052,0.0000,2.50000,10.625,29.3750,64.37500,123.12500,319.375000,44.445980
fork_bonus,15.985367,46.288377,0.0000,0.00000,0.000,0.0000,0.00000,150.00000,150.000000,15.985367
tempo_reward,12.334023,11.864454,0.0000,0.00000,0.000,8.0000,24.00000,28.00000,28.000000,12.334023
block_reward,9.073560,23.416336,0.0000,0.00000,0.000,0.0000,3.28125,68.90625,225.000000,9.073560
block_fork_bonus,6.712263,36.022279,0.0000,0.00000,0.000,0.0000,0.00000,0.00000,200.000000,6.712263


,mean_abs / mean_abs(total_raw)
term,
immediate_loss_penalty,1.005848
total_raw,1.000000
total_clipped,0.999723
total_final,0.999691
threat_reward,0.086349
fork_bonus,0.031056
tempo_reward,0.023962
block_reward,0.017628
block_fork_bonus,0.013040


,fraction_zero
block_fork_bonus,0.966439
fork_bonus,0.893431
center_reward,0.863051
block_reward,0.626690
immediate_loss_penalty,0.583585
threatspace_reward,0.545411
tempo_reward,0.366152
parity_reward,0.085414
threat_reward,0.011611
total_raw,0.000159
